# MediAI on Google Colab

Run the cells **from top to bottom** (press the play button on each one, or `Shift + Enter`).
Wait for each cell to finish (a green tick appears) before running the next.

| Cell | What it does |
|------|--------------|
| 1 | Check that your Gemini API key is stored in **Colab Secrets** |
| 2 | Upload `MediAI.zip` and unzip it |
| 3 | Install the Python packages (and the optional OCR program) |
| 4 | Load your key from Colab Secrets (it is never printed) |
| 5 | Start the MediAI server inside Colab |
| 6 | Quick health check |
| 7 | Open the website (Option A, no sign-up) |
| 8 | Optional: public link with ngrok (Option B) |
| 9 | Test every feature from code (no browser needed) |
| 10 | Run the automatic tests |
| 11 | Stop the server |

> **Educational use only.** MediAI is not a doctor. In an emergency call **112**.
> If you change any code file, use **Runtime > Restart session** and run the cells again.

## Cell 1: Put your Gemini key in Colab Secrets

1. Get a free key at <https://aistudio.google.com/apikey> (sign in with Google, click **Create API key**).
2. In Colab click the **key icon** in the left sidebar (**Secrets**).
3. Click **Add new secret**. Name: `GEMINI_API_KEY`. Value: paste your key.
4. Switch on **Notebook access** for this notebook.

Never paste the key directly into a code cell: notebooks get shared and keys leak that way.

In [ ]:
# Cell 1: check that the secret exists (this does NOT print your key)
from google.colab import userdata
try:
    key = userdata.get("GEMINI_API_KEY")
    print("Found GEMINI_API_KEY (length:", len(key), "characters). Good!")
except Exception as e:
    print("Problem:", e)
    print("Open the key icon in the left sidebar, add GEMINI_API_KEY and switch on Notebook access.")

## Cell 2: Upload the project
When the cell asks, click **Choose files** and select the `MediAI.zip` you downloaded.

In [ ]:
# Cell 2: upload MediAI.zip and unzip it into /content/MediAI
import os, zipfile
from google.colab import files

if os.path.isdir("/content/MediAI") and os.path.exists("/content/MediAI/app.py"):
    print("MediAI folder already exists, skipping upload.")
else:
    print("Choose MediAI.zip ...")
    uploaded = files.upload()
    for name in uploaded:
        if name.lower().endswith(".zip"):
            zipfile.ZipFile(name).extractall("/content")
            print("Unzipped", name)
            break
    else:
        raise SystemExit("No .zip file was uploaded. Run this cell again and choose MediAI.zip.")

print(sorted(os.listdir("/content/MediAI")))

## Cell 3: Install everything
This takes about a minute. Messages about dependency conflicts from Colab's own packages are normal and can be ignored.

In [ ]:
# Cell 3: install Python packages and the optional OCR program (Tesseract + Hindi/Telugu/Tamil)
%cd /content/MediAI
!pip install -q -r requirements.txt
!apt-get -qq install -y tesseract-ocr tesseract-ocr-hin tesseract-ocr-tel tesseract-ocr-tam > /dev/null 2>&1
!tesseract --version | head -1

## Cell 4: Load your key into the program
The key is copied from Colab Secrets into an *environment variable*, which is where MediAI reads it. It is never shown on screen.

In [ ]:
# Cell 4: copy secrets into environment variables
import os
from google.colab import userdata

os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")

# Optional settings (you can change these):
os.environ["GEMINI_MODEL"] = "gemini-3.5-flash-lite"    # if you see 'model not found', pick a current model at ai.google.dev/gemini-api/docs/models
os.environ["OCR_ENGINE"] = "auto"                       # auto | gemini | tesseract
print("Settings loaded. Key present:", bool(os.environ["GEMINI_API_KEY"]))

## Cell 5: Start MediAI
The server runs in a background thread, so you can keep using the notebook while it runs.

In [ ]:
# Cell 5: start the Flask server on port 5000 (in the background)
import sys, os, threading
os.chdir("/content/MediAI")
if "/content/MediAI" not in sys.path:
    sys.path.insert(0, "/content/MediAI")

from werkzeug.serving import make_server
from backend import create_app

try:                      # if you run this cell twice, stop the old server first
    server.shutdown()
except NameError:
    pass

app = create_app()
server = make_server("0.0.0.0", 5000, app, threaded=True)
threading.Thread(target=server.serve_forever, daemon=True).start()
print("MediAI is running on port 5000")

In [ ]:
# Cell 6: health check. gemini_key_configured should say True.
import requests, json
print(json.dumps(requests.get("http://127.0.0.1:5000/api/health").json(), indent=2))

## Cell 7: Open the website (Option A, no sign-up)

Run the cell and click the link it shows. The site opens in a new tab through Colab's own secure proxy.
It only works while this notebook is running, and only for you.

If your browser blocks the pop-up, allow pop-ups for `colab.research.google.com`, or use the inline version below it.

In [ ]:
# Cell 7: open MediAI in a new browser tab
from google.colab import output
output.serve_kernel_port_as_window(5000)

In [ ]:
# Cell 7b (alternative): show the website inside this notebook
from google.colab import output
output.serve_kernel_port_as_iframe(5000, height=850)

## Cell 8: Public link with ngrok (Option B, optional)

Use this if you want to open MediAI on your phone or share it with a friend while the notebook runs,
or if Option A does not work for you.

1. Create a free account at <https://ngrok.com> and copy your **Authtoken** from the dashboard.
2. Add it to Colab Secrets with the name `NGROK_AUTH_TOKEN` and switch on Notebook access.
3. Run the cell and open the `https://...ngrok...` link it prints.

Anyone with the link can use your MediAI (and your free Gemini quota) until you stop the tunnel, so stop it when you are done.

In [ ]:
# Cell 8 (optional): public link using ngrok
!pip install -q pyngrok
from pyngrok import ngrok
from google.colab import userdata

ngrok.kill()
ngrok.set_auth_token(userdata.get("NGROK_AUTH_TOKEN"))
tunnel = ngrok.connect(5000)
print("Open this link:", tunnel.public_url)

## Cell 9: Test every feature from code

These calls use the same API the website uses, so you can check each feature without opening the browser.
(Each AI call may take 5 to 30 seconds. If you see a quota message, wait a minute and run it again.)

In [ ]:
# Cell 9a: Emergency red-flag check (rules only, instant, works even without Gemini)
import requests, json
BASE = "http://127.0.0.1:5000"

for text, lang in [("I have severe chest pain", "en"), ("सीने में तेज़ दर्द है", "hi"), ("just a mild cough", "en")]:
    r = requests.post(f"{BASE}/api/safety/check", json={"text": text, "language": lang}).json()
    em = r["emergency"]
    print(f"{text!r:35} ->", "EMERGENCY: " + em["title"] if em else "no red flag")

In [ ]:
# Cell 9b: Symptom checker (normal case, uses Gemini)
r = requests.post(f"{BASE}/api/symptoms/analyze", json={
    "language": "en", "area": "throat", "symptoms": ["Sore throat", "Cough"],
    "description": "Scratchy throat for two days", "duration": "1 to 3 days",
    "severity": "mild", "age": 28, "history": ""}).json()
print(json.dumps(r.get("result") or r, indent=2, ensure_ascii=False)[:1800])

In [ ]:
# Cell 9c: Symptom checker (emergency case: the AI is NOT called, the warning is instant)
r = requests.post(f"{BASE}/api/symptoms/analyze", json={
    "language": "en", "area": "chest", "symptoms": ["Chest pain or pressure"], "severity": "severe"}).json()
print("Emergency:", r["emergency"]["emergency"], "| AI result:", r["result"])

In [ ]:
# Cell 9d: Report explainer with the sample report text (change "language" to hi, te or ta to test other languages)
text = open("/content/MediAI/sample_data/sample_report.txt").read()
r = requests.post(f"{BASE}/api/report/analyze", data={"language": "en", "text": text, "question": "What does the low hemoglobin mean?"}).json()
if r.get("ok"):
    for v in r["result"]["lab_values"]:
        print(f'{v["test"]:18} {v["value"]:>8} {v["unit"]:14} [{v["flag"]}]')
    print("\nSummary:", r["result"]["plain_summary"])
else:
    print(r)

In [ ]:
# Cell 9e: Report explainer with an uploaded IMAGE (tests Tesseract OCR, with Gemini vision as backup)
with open("/content/MediAI/sample_data/sample_report.png", "rb") as f:
    r = requests.post(f"{BASE}/api/report/analyze", data={"language": "en"}, files={"file": ("sample_report.png", f, "image/png")}).json()
print("Read with:", r.get("extraction_methods"))
print((r.get("extracted_text") or r)[:600])

In [ ]:
# Cell 9f: Medicine lookup
r = requests.post(f"{BASE}/api/medicines/search", json={"name": "Paracetamol", "language": "en"}).json()
print("Source:", r["source"])
print(json.dumps(r["result"], indent=2, ensure_ascii=False)[:1500])

In [ ]:
# Cell 9g: Health summary in Telugu
r = requests.post(f"{BASE}/api/summary/generate", json={
    "language": "te", "age": 34, "history": "Type 2 diabetes since 2019", "medications": "Metformin",
    "allergies": "Penicillin", "symptoms": "Tiredness for two weeks"}).json()
print(json.dumps(r.get("result") or r, indent=2, ensure_ascii=False)[:1800])

## Cell 10: Run the automatic tests
These use a fake AI, so they work without a key and without using your quota.

In [ ]:
# Cell 10: run the project's automatic tests
%cd /content/MediAI
!python -m unittest discover -s tests 2>&1 | tail -6

## Cell 11: Stop everything

In [ ]:
# Cell 11: stop the server (and the ngrok tunnel if you started one)
try:
    server.shutdown(); print("Server stopped.")
except NameError:
    print("Server was not running.")
try:
    from pyngrok import ngrok
    ngrok.kill(); print("ngrok stopped.")
except Exception:
    pass